In [0]:
import base64, gzip, hashlib, json
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("run_open_ts", "")
dbutils.widgets.text("source_update_id", "")
dbutils.widgets.text("silver_update_id", "")
dbutils.widgets.text("scratch_prefix", "")
RUN = dbutils.widgets.get("run_id").strip()
RUN_OPEN_TS = dbutils.widgets.get("run_open_ts").strip()
SOURCE_UPDATE_ID = dbutils.widgets.get("source_update_id").strip()
SILVER_UPDATE_ID = dbutils.widgets.get("silver_update_id").strip()
SCRATCH_PREFIX = dbutils.widgets.get("scratch_prefix").strip()
assert RUN.startswith("dq4_omop_") and RUN.replace("_", "").isalnum(), RUN
assert RUN_OPEN_TS and SOURCE_UPDATE_ID and SILVER_UPDATE_ID and SCRATCH_PREFIX
LANE="achilles_metadata"
payload="""H4sIAJjVjWoC/8W98W9kOY4m+K8Y9UvNANXTokRK1GB+mduuXTTQ272Y7h0ccHfIpSSqyrhMO9vprJ3aw/3vR0ZmZeV7jnBE2O/VDXoy7YjIZ9dHifw+iiL/j//nmw/692/+OXz3zXt5/PGbf/5m/P339x8ffz/+jm/u392/fxNDzIEjvXmg30v/8fbtW/3w+w+P7x4/fPn2jdzJ258/3B555U2w//unD39/+81333z4Uez5ClM7S81TSk3c45y9ZQmDatTM2EPoBDFKF8kN5owlFElzhNCgF/YH2eP++Zv/9G/f/+vfvr/527/+b3/6/uaP//nmz3/52833//sf//q3v978D34z9Kf/8U8fbt/+pA9v/t7fPL57/09Pfrmbf/g/725uvvyqt+Pmj3/+23e/vnAn7/Tmr3/7tz/++b9855/88Pggjx/fvYHFO19ejsdfTiefg8f/AT35B/a7DJ3y8e3jp9+wy6P+cP/w8+fP/OM3/+93n+0IO9oRFnaUOuqINIrMoVyosQLriJBpwigDeHKDms18vTUQ1K6TWduU1CjEL3b8w/d/+t7s+J//7S//9VLD/frfG3f8742L/15KqBFrLbWVgRw6qjRpDYeWHtMMs3MV5lltRY/Yi72BaYoQQ5HQv/z3/vHPf/3+3/7mhvzLxQv13//1T//9+7/e/EP47tu/3n986HrjS+Tb7/783//0p6N/wHff/he90wd5++1XqyPtiFZaoGX7l0cPLfHgHCM3zkyUNdlqGNpj67YSRk80ogAigcCYpQTUHBKMuglaBsKfP75r+nBzP2/e68OH+7sPz2P23w4f+hoy3BEyXEBWp6qmNjiMGZV0JNGaE5ZCc0ofiLNyLqWCodZLGINZci8RkKi2uQlk8QhkN+3nmx/0bujDt999++mLN/3+ruv7R/OZ316OJe2IJS2wZEQlMu/TtQctWQBC1TRKUakYK9Ak26wGqwonLSLIsVAKqXC1NzbBMp3A8meVw0vt9uHxR4PUv39zP998/v5iOPOOcOZlzI4QBHLhOGONVQ0qHjPrNBeYumJIrdjOzYBjdE6hxdwKdYqshjaUTeDEE3A+mBENRf/rZcuy7IhjWeKYh0U+LNliR69xxEEjtz5LV/uuzllyrog1TcAaiWNtCFypUgyilDbBkU7gqI8/3t3228efDcwvX78MUd4RUV4giqQhCE4gwN7qDDBlAmjIbUyMCppjA6Wu9qkxUm9i7FOxybDdDnETRMsxRP/n7eOPN7d3P8lb45XvH+5/unW/eQrGwx/hGJZ1RyzrAsuuGick1VDi0Fa70bXMo4Q2KnDWRqlm0BIN0koWsy1856AUwKCUuI3T5LNYvr03+nt7f/cCLGFHmQNLmYOJyiQjjAevSMaKDcU6WCd0iz+23QkL1BZqwRzKLA0KEZccQ8sUtiFA9SyYXR70zYfbR30JmjuKDViKDRrSCmPopgYHZgs400IRQM4GXEwyLYSjBXUdrbZqPnWY32QL+KnlFMo25NtWz1dwytu3J4P6gjGtwvsFDOqoT4UdpQ4spU5PTTtJnkMM9ihk276Yw4Tm3iAmEhZbtybnMmKM7hyM6Ce2wNbiZuR9wd7v7u9+Z7pB5YOO3xb2HTUTLDVTpgyzdQVb6rlKSb0ap4mjQANzsMGchzad5iU8b8LKc5rbJe5UIRbehmVBfIZm3cjdWPCEJ6TrMuZwHOkdpRYspZYFOFMIRg/YhKkhOkswRzL6GGihzaSVDhOtvTQiVDb6FWqgajJfLeCNoRu5EzgBtfyg331y0/bVjTzezNuHD4839+2DPvx0iH3+6dt7x9s+cQUxgx0lGCwlWIilc0yxGKENphRa6X1inqhg8Fp4xMkhmDoLHNR+9NBUmYBHGJGL0kYYPy9or0X7iBM5aoHj4O8o2GAp2ErobF68G2GzhQtgJmhNZu0V26DJ2uLM1CXQGIV0TMnZ/E0zPYditG8j8E0B/+H2w+PDbft4wNHD5jmMn0vP/OWrz/+3T5//Gt8dhRwshVxSQSwzB6zU2V0ClDrLFG5aRkvGlWPMTDKCcWQLljCQW/PUIGeIWy1uvBrfF+ZyzgC/o96Dpd4LtWnt2PIo0zY3qlbVbl93w5UtUFpctDjZI4aRZhUNODKbojF6GHPaCnjT0H/Sux/MZxjkX6P8D0N+/vCP/uouK3xHMQhLMRjMK/Se/dxFYyqFA3FS0yu9J4NzGNWWUCyGjRJxVIuohjhWmcH2gea5EdD55UDvstTjjgoyLhXkUCrmp8VQTSFynhJVZ0euueQ6sGNiisJpFHM/krKkkIzHxAzShXEjC5TXWcD9kRH327f6KVR++eZl8O8oOeNScs4eQjMSQ61zSgwgFjKHTmfgqXFvs5Q5Mxs7L2YC1NLFbIK9WFTteSv4+QR/eXvUJp9w/87k/U0Kw/7qD/pO7x4/fLtE9rN9Pj9k/dGXmWZHfRqX+hTBfNPE7ms/h4511NwsvPYkQTiLiSdjOKlSo5FNsA7j76XnpkZ2EtWtfNPp7Iq5mcfbu4/3Hz8srGNWUXvgQbSaQboY/sP066fvX4b6jvI0LuWphYAyTApB1W5yyMgPTNNIItSmkR7uGiByGy2mDo1jZ6Q4nO3UMsdWqC+TMNeg/s7e/vFr2D+/8DLcdxSrcSlWKTNwmoKtKY7GZDSnhcKxQu8ybUGziSbzOKWTsXvz/cNophHUIKNb6N4I91Ni9Yjb//AoD48b472jcI1L4Vopq/kXbjOOBGoSqQ/TrqabUgpGeComaFRmhVxKKpEpz+nM0wjpDLjVOo+X423wboz2jko1LpVqH0mKMZisyXi6LeiOc0CRyjU0zGDvJnMrQergaOJUIY5QsYzY+qw4NkL71Fnt3ZcXn+LuQfXM+y9Df0cdG5c6lgoYW+9otKYWg1rGdFy7yVdGYz1aQqeQy0Ay2tNIivGgDr7qPee4UV4d8KRPP0Yqdd4/6CFo/u5+/u6ZU/Nfji7OwL2jeo1L9drriCFrIYmFc0YKM6c4apfGTMFXd5rG7j0zY/4lZFHsZNErdzLCudVip2vgdufyLye9/KuA31HNxqWaja2EAUYYo4QaM2V7vHmeWNMskzUVqt3M4FmFYb4G02y9qWYZKVXzDhsBn08CL49GxsVV1J1JJPl57VG+po5fpS89H79QWCtOeSpheUSFnXVLaUfpm5bSNzBUHrlL1sxSKNcoWBsiWpgdaVArvUoqShNzUQu60QJ1zTS4efptI3OVV5tr06icdhS/aSl+ddj67wY9Jqwi5qCK9DBQ1c9LZsjRvH828BWUubaUp73VvMQqpiRtIwMsxO+RALsqsjhY6OxB9hmQd5SxaSljI43q6eE0py3xiaZm44CclKrGOaIt/jFGTmCRwPSuhqRoMXeau8LZylbHrPV5kG8etN8/jAMb+vTKm8ef369O/U698bJ1vqOmTasj164ptJ5rwTJjGmaNGaiNHphbRDaVlXAki75JWUzS1iHVDEY1mxjjjWouQ7jM0fx0++HWvur948ODGsjfuU0OLy6t8fSlE3b498MD//LlgQsr7Khw00rhiqY5VKvkWdV0FGXCCsaCVKgFgF7AwvKcXr/JWDU0TYIlmeY1LxQ2ssJC4a6x/mUb7Iv5jio3LVVurEb4jX3KoMwFMY8wzc9rLJH6sB2AEyCUQQw486TsQreMDoGNKG1U0hlPHs8+gf+rnMLlJjgXd8+ZY0cZnPLqtNw4vnkWbjK9mEZNJqRQUzQlUFqwP8xMMbTYRughp46p5SIax2ihhq3MsZDB4/bD4+1df3xqjM8Yf3CLfbba88daz6K8o9xNS7lbBaUiFDbfMXFokNlHz8IjSTIDQERz81gzwuxECk0yRWM3ps3sU2kjlHFDd++vLUj/k+KGX8XBcxvkGr1wzqA7Cuq0Og5OgaiZklaGMSXxAD8bGKbzRi81YQAKXBWQo9Iopqq7RNtBwtBzkI0Mmo+fw1/spi6qTXsW8h2ldFpK6TJqaq1buKYwTAu0XJuTU/bitVFMD1i0NooktomyxRe/b2G+i3rnXDvFjSAvT4P1L0T1alFwBlzcUfjiSvhqyaFAN3EFc+RmrsdoaJlsMhekiskCzodIATMEMYnQi6LFi9BjL9w2ArdeCO6F1cPn8N1R1yKsSiyRSq2pYp/Fi62BpxE/nRQTd232Pya20Jp7xOQ5olktMpB9XYMZZBt8l2dYn/zBU6Z5Y0h8uB16Izef0N62lgR3FLoYV9kE6YVglGQ8/1CjMFC8zpVMTNXpJdrE5joq+7WCbB6Dkh9t1WEO3Pz3RqjDmVXtmc4hj3rzL5/Zpn/zfC7h2WW9o4zFpYwVyGBLvZurHYOkeFIzA0RTlqwTe46SchZb+YIcssXDStq0DUydh8JGAMdX8JpLOcyrEpzPmmtHvYtLvWuKSs04Rilb4xArojqlt4CJYYSCxbVXGkY7M2Kn2vLsAg2NuBBbaNjIXEeqM3+tKrEd8POeUhd3lLq4ugwaI5VmtC/ybLF0DLWEnBt2i5qqBnHm3Ezngt9lmhCJ1atIZg1BVbc5dbG9eEl64Zz2vTyX/Cz4OwpbXF0dHSm25rmFMKMf5Y42BKdfZB4aUxSeal4ImQOap0oArYZscrg26Ywb0cUIz+YZPgF8VVnOs/DuqGhxqWiHr+CZILQyRxqVDTdb5zXOFNGIOhamVqo2KEjD/vIL485v6gDZqijH9tdFhOaLO/lw6GFwzKsceeclBthRgeLqSBfZM/UIg3Rk+38iLczqCYRsztvYDY5UZ6rgFbUWbWMpGkBwMBgl2sgAi5PFr/Ae9+/k9u5ZR37MGH28e/Mo7a2+Odpn4hz8O6pRXKrRGc2hj+gOHWz9I4LQMMduQdV0f+6UpFPQJImzUjfPk6Ot/ZwHW5TdqEg2pssI/e3dgc+/gM2fPbaiHTUqreqS64TWJIDpzprEvAxobEZTeJiWSjI6akolgEn+XNosLfjtTLLVLob4Rho1+RWbh/v39w+/cJf3ev/+rZ7im0/tcZG++upnfLsMIV/9sNf9hCcPtf26fuC3V62FHfU0LfX0GLOb5gCMcULIRVMuU0PtuUJW22u2CLJgKlm0UspxzjksRNVJHTBvlFBNcb0WXqWpT9n8dQ89/5zrjLyjfKelfK+txTBkSEvqPY1QYjfnatK9JxGJU2iOAbF59pVNoBgfySmZ0oxjhrLNnZu0OiT93FDhXD+fzx9bILejLqelLudWevTWZtyI/TBZu0aZQ4IMhMBABlPqxXQfNmzBSELNHJyzgXG1jdrQrO6Z/oKck4IP700my9vHn28W4f/Lyxe1/ziG8Y5imlZtk9KcYtq4xt5DbSD2eONjWGYIyZwMgzRN2LREJwhzqgUq6T2yabzS6kYYp+MYf0XFjoF6KOA6wsyOG+C88j5ljh3FNi3F9lQqqZFGjAImRQYYF0BJpfpBm7i+m9UvoB7O4USzMWdlsCAxa1XayFksBckv5jhujWP899l3r9kHOwptWgpt03OTDdBE6qdbDHOW2cIAo8gRQL26N/qFMaPB1Ks57uw33LNmShaZt8ly4KWlLIbnuP0UrVd5wF/eWAmSoy+fsMR/+vLw4/qEdtTntNTn3h9HtQWv9U1pUoTmnBgsUjZIGXViGP5imqjEg4dfsqy9qJqCiXMjqywiwDHsF+Utv4kNdpToxKvmMSURR1MphqpUo6w6BysymUvy2x6dTIxLLTOaZcgihyl1dclO0bZL3cgGp0pdjppjXe5ylUkuKHs5a54dJTwtJTyR9Gm7Y/aqnkCBKBbMqXofyIHiPb0UJ+Y+ZsMcpXMLGNVCvg6smTYyz/HSl6PGubr85RzaeUf1npfqXTIxhRC8MR/7HYSeSkkaq3gTvxKGRYTOJMVUFRpnTVFnS37LNWg36bYR2rhDmLjqGOnMxrmyHOasgXeU5HnVKitPlOm9x0yDU8J4yMZob20kHBbzxW8yY29+5NFNgmD0zNnwTrZNN0pIYqBXR5zl609rjs+6xHPVyJfYbUeVneOqZSn01MYhNx9GVSzeOaQeGmuVUCEI2s8zbpAERp2NQs6Sagi9t54LbmS306VM14Wgi0qazsK/o1TPK6k+vcEchNZVopaZuIfI2nOsIZuylNApD3sppWg/W9PQQ8cLk/Ngikc3gr9eum1edPXhLN47yva8ku0s5C0lmy12Lcm0oPMzv+jp17RKNjcmKQQXi5OoaznccbZwBVB1lo3c1LIS51m8f0ny7ZG9zzvq87zU56U3diEufZQyw0TMyWJ7zr2ZgCyVEiQxeRhTZzTCFaqRs+RtqNFEvcpGuMN16/wFpTln1/qO0jyvzsBLbWx0VkKxx7Yi1dCeuTMniYDsB1Ezi3n3EUPNyXsY2a8wB5rjF8gbYR5f6Fsu7V17FvEdZXdeHYtHW9mTai3BnEpU40DGZYtmY7OlxWCwlgZNydy9xdpcMqZRNCnUBFU20hTLW+WXI/4pzfdquHdU2HmpsGc2gd0AUxwwB3kzOc7U5vTba0MQNZu34daMhxqjMQLqWXDGPme0lb+RqFheI7+Cc5pLefz44RStOfLuS2XejqI6r87FZcaBLfRhSrlEDWiqLkCGWGft1M08UlMfFnuhIlDQEql0NAkII2/UAB/hpSrgObZ/6SXDc9YoO4ruElYpjpiK+SCa1ThPEXVS462fib1TF2D267WUu1+1VSXOHTrCGNSj8Z2tgm7ean98d6Gpnt9YG+izsqOuLqt+YNoGMY5YzVIhjhSijKw0pdkewsbD70OEFlMQCjN20wiBp2iU5m0HtrFhvJywXpBavLya8KwhdhTKZVVNXqbmFGsN5Hd3kWMcc84cpaL3aqDMTIoyOpgLTBrEp0dhRc4K3NtGhjju2t6cMcTxaqvn3n2pQXaUzmUpnSfxUJaWm9fZ1iBoBCDHTKGo9tks/tvuaNVMEtxyNNKYKQHNnpLRho0MgkcTuF8X6vT7330yz+3dD8dNdjQP9V5uvc7g2+sSIA9y93/bz/nUI+jNp1/j8zddPqg98AIOV3YU5GUpyMc0pyah1Z7QS0ojhNClQkjkY7wKQR6xe8/xVLUgztQnBi6Y/US9b3RSlcLF2+pzNZ3sosjLjoq8rBR5AYv5teU6Ui0a+qzgkwu8uQbN3MxnjRo05dZk+BYTlSw1Ri+lbhv12sAr6umescqGNXWv/inH6uqOPfSqsquyY9agrLIGIaWcWpsYQZDHoV5Ijb/PXqc3N6Ym3jskNvS6+So9Jvs2zJmMdtBGB/pPa+ue3ZSvsf/rH3zZs64z+I5Ji7JqxoYlZDIfq4ocWiox9eYt8y3gxuHDeSh4/2tzww1SZRAqubRAg7zIdps0ET1TwTFUfjmOlo8f1iTmyUsnmMsf/CkLiHdMVBReZfnTGNknlhh7z0I1zgyhgbe3npJHM9caose0IfZRsPeSzOLTuqCWMTaCGI5V6ztr2QngHfMOZZl3wFYPC1JqKqydu1GIGluAUkPx4VsWz7x9YIoyecwaRUL0Hl65hpqxbwTwqVqLA8DXap8nePKOmQNeZg6qSciUvK0uG8sWNA424xyEuZF4HYt3sUhhjBLLSOIdkXtLGbOt45Zj2wjPZ47rF4v2N7jD+dQaO+YAeJkDYAhauGYv2y2KHcYwGXOY7cUxVMRYmzmYntoM5OXvg0EVojkWwZi38tAL6flpSX+duDm8ciQvc+L1i1f9jhKfVxK/e4ddAlQyV1GIUjFFIhYCsRnVCTJb8cw9szmRGrxJGpnWNGY0AaZsterzyfEan0B/2YCBp8DuKNV5dcrNJvsssGWgjqFJqT3kass0Zm4tmKNxId9hDq/vNzYiIUeSXGeBFAtsBGw9uYBfdKz9FNEdZTOv7nIPX6e9aKc+ja+LUbUSmDGj6ehGI1bz1j66wVSd4c5ksdGi4shDlGEb2UzLc+wlonseXPOOMplXbbkrKrAUQTFRVIxeuFc4VGh2k8kwvJrT4qExje5MIjT2OxRTqzHlslGrH/KD67VPeLx9pzfz4f7dZ+Af72/eulb+okGeLw58unp31Ji86r6dTFFALy0KCyAn0xoNuq3bHL39Kg5zBn5zHkLmGHv26mSimKdKaH2r1RsvB3U8fPzhWjx3lHC8lHA+Rjqh8VmjaNGoGQcjDAVHjOBKzvY9z9JzzS2pqQqIRiHm9FNq9dmgYSM80+V4Hs6arwV0R8HGS8HWogV4GDhlQA/KHahgmy119CnUyW88mD+QjlHDTBR5gt83rCmVNvpGegLwckDfP9wbM/n4oNeCuqNI47qacOD5dzIfmdDx8w5plE1mCJK0In1MbibLzCUkUMjeu4EFe+wDcp+6Eah0OahfRa0rYa07arW61GptGlYJwPa4KQKFErEWyqDV6EBDBqSkwdYyteodGWZVNH9hgjhHY1wbkat4Wh38mlg4egp16p2Lod5RiFVYQW3bPZWc58RCNfRgLBU6ohYf/k0mC+KhSDYVZK+HBlaMoxOYakPeKI+TnmFdu55O1B2lWF2VJXcAP8runEhbLT5xL/XJqWI090FQvRmyCKVMVaL0Ttm0LsYmiQvWjZC+/HTiazNseBxx/WOPnT8cnnJV/rnuKA7rUhx6HlSnSE2KXsLQiVopFnfT9N6oknKT6Y3hOYCK7a5OzMm2GZHxHeWNTP3kwGG5sR5/lM/NkV5l3V/36efnX2nTxS91nUV3FKd1KU5NhAoV8ubmpkNzKLWWOpy9x2oy1KJ56FTNzLaz42HqTvP5u6mb8J+5bNOeIV96J/QLc1pf9vnyxqrp/NGXT9/O/fzw4yftdUclW5dKtkON2JvQtCc347QyKiuhGYO1D1JqBLNHGtGM0I3jGokwwTux5WlqdyOrrLsCPMF+kVT8TWywo/CtK+FrTo2arfZRs1uDayQscJhXfegVBDp09DSKFxcZ6DVRqdHrizzvCxvZ4NQ5xVFzrO+EXmWSC+6EnjXPjjq6ltUxklgsypywRWMVNEyYUJoYOcXsbYWMTIfUy0hpRu5O6VhVehux2JbaynEdvxN61DhX3wk9i/aOIrsuRTZ00pzDmKN4iUFn9DGaKYkFDNRcCSD0LKFbnGgF0BsJNJPkzQS5GSBshDbuECauOZY6t3GuvBN61sA7Cv66Gn81WwpTXP2IjJB7NV7HblozJsowo8IEZE3eeyZZaDLmRz5JaZqvA93IwPTqiLN8/cgcmnMu8YKa43N2g7BfSsGevawNi1S7D/ws5HP5YrQtB2Y6qqZqe0k++tys6ORc/eacvehzPWIagbP0rfzg6Uuh18Wgiy6FnscfdsR/mWc4zKAJxWs/EvU5U00xFJJsYYhTc65QS6sMicOoiVNmGVKSNyCEUraiavXSjfOi47PzgMcdAV/NxAoNyeKMqI+bSSyZNLY4g7mpUBLY0jeJ2jEPDhaRQtVooSnSDL2H2bZpF5shXAz4nqdrENKOwC/F//D2ZEVLGxRUQirNU+0xQS+cajf/0+PE4WofR/aa+k4ztVZ91vbYSpSsWohfvtIvvaN4fq3jjpAv1XmiamuZEWaZWUPOfnOuphpqGJFrE59UUnDGkVkjsJliuKPpdGjdixtBnl4G+WWXFM/jTTvivRp5lYbCNN2XjdRYwBTh5pXuPphEFE2Bm58JHEhSMv/uc+TNq5jrCdQHbnTYmePlvuUSJXh5Gdt5U+QdTZFXAwtKgVZHLtqmWrj0aVYKtXIpxds0Bb/j04FTGAF1mKc/dApPyTYLs2xkiuOE9MQdnl/fP3Z68uy7L7ZI2dEiq9FY6h3JMGB13k8QudTgk1Y71Ry9c7hnS8wBZe8xZySod8JpMrxkYtxorE++9hLPUZsdFQ6/XOK5jrBedInnvBl5RzMupXwsokwR0GLKNAcopgcoTCDu1QRfCNAseqfcWxwszegTEKSK3DLYP90or5XCxRvrNedkcJ5A1R2RX2rsXEye5RkhStURqeSepbGwfXLU0avRp2isqZk1Rp4jhG4abcYmA2trYyPkLz8oe8YsG56bvfqnHDtGO/bQq85gAHbU8BBWtCPCZDHpYqsAy6EDIYSWvXm6sa+WJRE0mrYKOPrtHQuFRsW9aCh1498bLYwnx2rPbsuNjtaO/oxrzf7M73ml0XdMHMAycWDUssSOFh1Tmuy56hrLrNjRxCtiraaxjPIPTWzC1tRUEpoUYmT/oGyUuKnuDdTYyN2j52u8wOb+Ud5+Sa4ezlJ/lJ/0V8/wH7/ifVG4PJHqvoTdwI5pBVimFbxTl1FH7hYOp+rwk5w0WgPg7reBTAyU0mNSklFgROXYckATybYTS9iG+pdVle7Dxx9u9D/e339wjH6TfALsmE+AtErkBIpDTGT5fELNaLgao0TkbPGuonbsGcVHR7igdahTjxMmZnsNtuGTJT1B/M0TxHet1AHYMZ8Aq1L0BMbhOo6afMSJEQych5EclWuUQGEkMpx7Ky0jxFChaEzUytSCUngjyK8o1Tlijy1Ldl78+KOlO18/7cq4s2OOA5Y5DtPVIaRI3iuvh0lCjbrFGlPNdfYOsZXOzSgqJyxtzhk7maaLccYy5pCN1sDTGp6jW2+rAp7Fw68t5Dn2m11p3x0TJ7BMnKA9S0x+hwZJGLypFnkxfoOSSA71IkQjN7+VYsoiWQQebQxS97Bho/uAfGlFz9fQrw5rv3prSS1OvXFeAC5tsmPqBJapE1VTcmqbqRAm7/VOGMpsXrYQcvNeW8b9BZnRb1GFqcFYofEQKHl6un0jmyzqeY4jvzhf3d0CO2Y9YHVLoHCKHYi6KW3F7B30R+21UTPG5/MFJ1X7UM9GNUarA9V7BA2/48YVxkYWOFXNc8IY63qeqw1yQU3PSePsmBiBZWKkq1HpyVy96582C02mg0tKo/cBkjg0tm1TmaUE72pM1ELyfvx9SBs1b2Sc47U8J0xzdTXPKZzjjnmGuJodHmdqSnPEJjpr4aFGr4G1lt4iGP2OU6BisxjhjfXBNBHALDIBy5C6Ec64U2i4ppLn/Ia5spbnpHl3zCjEZUYBNRppq96gcQ7Mw+SrZGUfrgMW6HWWUEYX9jKuSM1IHTbIXMTbnNa51TaiTaLM+p2nlTwXOcMLqnlOWm7H5ENc3WZnwpzL8C4Y5u9aZ/eEmM3P5dFZfSJDNE1Gk7M084XdqEJo0V2hKs6NLHe6iOf6wHNRIc9J6HfMQsTVffdq7q0GDU4EJLZaNBlDblQssohXNyTunUwvz2BfFR8Rwwipc8050VbEoGyzaXxY2Me79aywk0Z68tErTLRj1iKuqiAwGRkAr3Ng2xESa7DglUollgxG52pIKXQNlUynzj4opz6ieToTOnOkjUxUT5noNZVVJ+HdMSEQ18PPW8rDqwezZoM5JKO+fiCYmQfUSmXOKh273/4Inooj8I78KLkIb3QsxcvM5zF4d817xh0FesyrWurqTdtHYvMitrAtEmdQo1yxo8E5os8qrmDOSGX6JGik4utcTM37ReeN8I7XLedLy6dOLugd1XYsqzkpQ2bUqqPNMKRUcXmns2WYKp62H94sptFss6j3jOiVR+HZW7PVvhHNXVZNnQX4smKpk+juqKQjrzrvGLlEcwVGHRmmhUAI3osSkobMKcAQKvbaxG6eGWfoOdTeJ+Tpo4U2YpnLTu4n0b27d3fxUW/+4e7jO9v1/bsb5zd3P3x3c//wi4b7xxcKtx0Fclz1ag9xcDCJPGFo5hlHAyrk/UAN15qh+0yakU0m5y5qJF9NGHubKe4iG1124WPX8T/D+gnl66nJS3BPOwrmtOrKPllq9Q4TOJEatiKIyeRwCD4KYkCa3tNYzb0cylypxewjPEqdaJpqo7xdPBsa1ymkF1X/ncR7RwWblgq29phjYfsLMI0OhJw4wQil5Cbkt0+icjUhZGu9m2ISssDoM5JnF5wbUZF4LjSeW+aHzfBGPlzIwo9+/Ar77KhTU1z1HxVtYbaGYIpVSzb/bhaqCSAwZayaQxmoSD4tzlSUWYWGqJaRKWx0J5JjeqV9/v7RQu68Xad1Thro+OevsNCOcjatbuiLqViLDKjTXBP5xHWMpfgAiXm4KlTCwFFkYLENdqDwXWyjscbq2aCNLHRlkeZC8B632y/VmddnIi6q0Dxpuh1lblrKXM3m/ULFaHy/cpsWagTK9Gb4bbDXQwv7NBw6aLBaOsU5jGZRHtk+sVWwoXPBZrW3jpU6n3n/2gC0oxZOSy0czHGZIOsNp9HcaMQ1SM2SvIUUtBkDYy3GvrypRch6GLloFsk8QgwWizayQb7ABseDzCWx5CzeO2rhtLpkX7XaWuZCBvAwUdrIpJsptmoyQ/MAwzrLIFPBWttM0fwaY0/Z58Nz3ojYxnIB3k/TaodXXgn1jqo4rcr3h5NYDBkxGpA1UawdvL9EHkpTY+mte8V3NiUxU0qDLJa3wkSBWtim3IrTWS67b7FV2lEop6VQLtq0x9AK1DB9MEYbsZk49hY5fRqT7YV6Z5MQc/hgB6HpReBjqDKFvhGZvaLY6qk1Niy1eunDjxVanej1doH1d5TsaXWhvsoQoJggW2BvA02ixGpuzatLZ6istulmncOrHXODkMqc9i9KLzH3ja7u8dMyq+cyq68sslqwuOtse+S3us6yuGNSANen6NXHn/rQwmaRv2WupbU8q1NpaMOLVrOZETh5rzMpYYL3Xq8JS0xhm8mR/KLC7a9gvphLny7fPhXUcMeEAS4TBp2FRy3ZvCsZY5gmPwtBFG+HkBuHKZI52w40GmFKdZosLa1C9mvLro82sUWFIzXED/LbHFzgjuof46qGqrvqaNABRmaK1Grvh/aJPImBhDQnsoA2Kiaf5UkFRCCbsqTKG4Gdngd7X/qAOyp5XB1Mj0iJZaJJvKSI5lDKAFvi1QvgJQ+/GF5m874TZhjjbUY1/MKYSUUaOW6E9rW12l9MsXmZ9pVPPl2h/SBXxpYdcwCIq6GsXvipCj4Vcmqqk6umZgEFWvTububSJmoySpGAO3cayTajd+qTiC1tZPQTxdlHfNomddn23BeVZP/6+1xp0B0TCrhMKKgRO68wSTVKizWbAbtZ0jRsCzP5YIVIlIRAdGrtpgFGRmMO3mKjlLhNZQ+EE1Psrzbpa1znjlkFXGYVYiyc1DYLGqizAgTKWn0sAFFJKRdJMXXvbVVSzWkGnjX5tIXQKreNSuDN8BeAvnO42jG9gGXVLhFNvwQoBqaSLXFbdINaqkYIULiXrtP4b7TXsGaeaXJuo7UmFVDzVgv9RUMKdwlaL3/882MJrw9fO6Y8cJnyCDVMyq5iyTabqVhKfn6XjLoAUKlNpuaCtkYyYg0dFAYHL7ZX5TF0q0XwzETCHYLY8uEvHkP48nC2Y1YDl1kNUW4idUYjnGS8c+Lg5AUgs06okMLgaGp4YCJjL8qlKM9pijj35lV82xgYjt8uaj/fvL3vn2BKvxu3P9w+3vyv2/cG9+K7EynaP33+pwtoace0Ai3TCnnknND4g0n76pVg3I0DxqA4TG+pT2LoWFv0g7oI7A2gjAxWP64wwHPbClo4B62PbPcDtM9/XwXnjpkBWmYGohe+qzbQmDOM4jeGmZpyBEnSYapPB/NJmhKHhST7pAWuiPa3sYSNhgEBLG/8dHnQmw+3j7rPYt0xFUDLVAA00RohHSYJGv1qpkoz5ZwHWpwXkpQTGc01AjAN2qDBJGyoHCwCYKa5FbrpAnRfs153lPu0lPsptdB7CVSnJMgpR5ashbnPRuI9IbzDVUBvwJdSNb0gtv8jIjS01bxNcgXiSc/6/q30Q5rRQ9FtP3TD9Zfe3M83n1+6aCi92+ivZqMlzjsqbFoq7MhQGZMRRXOiQ5oUb2Y8NAWk7umsmJMFrmbctJtHtnA2TDvkKtFlg8BWOC/c7KH48zeAeUfdS7SKZhWM/k01HUtQ5mHATRpeJ9fLjGkKElHuJo07di8wt8XMNDuNw93krWB+xv3uC/WOapdWatf4wJTYWvVW9cO+mpNLOlSFIGtg8yXaSUz7tgaj5zwD9KnDRG+XudmKTpetaC8H6j/Kww9645ezfOroLy+8ebx/BeI7al1a9Z6vLYXMxn2hxepsAbF1Mh7RhmkcC4gpCXoJrs6ZpGkgxB60JInubrZBPF16x/5gi5uhj3L79lAa96nW/NMLq/KRU++cMMW/Hx79h8Pnl9bYUXTSUnTO2kdvSgasaY0abH0LkxkCJxgFcRpNpQ+RhuboIVSs0ZzNoXpUzdtsZY2nHv0z5ouyxN8C+x31IC31YMrQCxn186xPM2xpSDm0/o9ZbMXnmdFnL/g0Q5FcyOlgMf0II2baqHOZYX/qXv3CDOvb9C8wxQX36U+aJe+oJfNqzhzILIb5wBhbioQsJfcZsI0QCsxIvbE5MRkwU+6qPvOsT3NYYRqxLFuZ5fiN+oVRrr5HfxrfHcVlXorLPL2DUiajjdnkIqCteu4+Ur0QSIxG0PtELj14rwng6AU/sQWvJxxUYSt8cdsAcM0F+gu2yJU36E8bdkddm5e6dkxKGrXlWEr2Ccx+oVTnAIsUxNJiaUaipl8yrbmkCRpT9iKDNns1Q29l2NM3sV/itC66i30a/h1FcF6KYKQxo2nbaXgjUsLu1zygNwBv0VJGrmy7CGPvaagPxE3Cpswqh2GUazP4y9NQ/ppLvqeh3VH3Zlyl5otXkweEMSZkMXXW/LJpryn5qFwhc2LdG++lNkhlGK8lSFnYAje0zaCtT6B9s2RJS4Rdsr1xyfYKkHdUvXmpestkW79NwbUV5tkMVRhk6PpgTDVngdR9hphpMS1jeJvJEHtMxUhtaBudf6Tlae9RkHc+7M07yt+8ahRPrfQsqoBGdIznmy4S0OGVtTCM5MfB3XPABNVEWsKgNEuYJQ3Otvi3whwuW9jmim+GPOrNv3zmo+NkVvL8yt5R8ual5KVQzd2GRlo5YkDRbrK32/K18NiNzld7X6MfWkRhYzwpcPIwikYtKW62suMLGc+ltOZF5OV8CN1RDee1Gi4lGl8NjQksNkILpsMC1KTFi/zbADSuEsznV4GoZtPMGbzrtE9B3mw7HBkwb8j+4GNUp6/8n38jMZx3FMN5KYbJNoOxE5mmekPmCRZITWgZ/n5dkry3HzXvYWsy2q9zRWSOXYOJgxIJtlJdMZxLRDynjC+/L3wS8rKj0C1LoZuiap6dIFDpPHqmEkzrjuC9nJGEs9pKz7UnjC3WYaIseluTXipMDVuxmgjP5h8+wfvZo6w8zNXg7qhyy2q0mV9hJ43SjKcgJBEJ0Za36MAiRmaK8fLpbSojmhIytasztiZdsucVwlbg0tnIunYlx+4knvvA1YbYUZWWuGqAFGVoNnVkQKMhTaZIudYyBQhtVRPG3KiKLemWe1QelLtImtN7+8hWhsjHCmAM+3H/Tm7vrnDox+zTx7s3j9Le6ps7eadXKdSyo0ItK4WatXEyt9OGIGMZ3bTR7GiUcoySnAhRU3P9PSduUEeP1eho9fpPTJuF13Se4e9bWVh2VK5lqVx5ZumeA6jgfFMqluINaITM79SeTYQZE03B4Gfuttwn+1wFCMUndG107cAgv7yy8Ig9NiwsfPHTj9UVfv2w66rOyo6yuixl9UCjTSIKGZOYDyyFgk8D41hiz9jRBDcEEKNf3j25kUWjPCmNMGlq32zTPSkrfLG0vqCqcPHsK+177Pe60ro7CviyOr/u7FUYI1bEgTgbesfLyhAideYWo4RhMuZTKj0nU/KtxWBSH4f3zdzGuhd3LH9nX3180Hd697huS/vVW8vYduqNE5zjv/768aVRdtT7ZXVbHFPrnKoZhBsHoVqjsQlJuUAvuaceDmcdHIaAEvmolNard5OfpoM2M8qCWB+HfnG8ursJdlTyZaXkw5hdup9kl+j3e7J5OCLxgevNB6FINo+o9mOxshg9LD65zYzD1VuDha1McOps9YQ11qesV1vkgjPWk9bZUeyXVc/y7BdIpEXjYrmWVnxeF5FzEaoTU0XqGlkENZL0PqQOBW/wEqHoRmP04FTT8nd6zDRXn7Wewpl3VPi8KosuRWvJIZmy9CZGVQ3jMqfPx2vm/2eiXrq0aoJ/zhCCxIk5Um4KbIRhK5zx0ujwBfjLosM1ycnzO+bKI9eT9t0xycCwHufdRGVgjy1adPdiEfvLdkhsXMP021kmcnsyFZCxtjxlTnN6PoE38Vbp+1XX8hcHmvU7T7uWX+QOL+haftJ0O6YleJWWYB8nlQb1mEs17t3G9HFpWiJp0dBSqz4EPJcQCldn6+SkwWKURSndzHSnD8uvjz0XHZWfxH7HPAQv8xDEOJo09dS9TAspRg1ABgBYDBqpd6hqrhGkTPORMfDhfFehNrWvdWyFfdlm2xzvW37SShf0LT9pox0TF7xMXJi3jTI4lUDDAojFpWqEdkqIoha/uKCxhVq8Y8Igbj1pzS1L91q5rBtNE4dV4/KvbfSamoaT+O6YFuBVWiAbp8ptmkiJHb0RK+bWArKOCtrty8jhMFI15ZRqMLKmxsyKhZwZiLeiYMvT9mP47toAhHdU6pxXDUC0zNZ9LAWlOpNf6Rz2VSyjcTR1GKszYmgwg9ReEnLygeziZ++4VaUtA1y0oH/JlXxuAP35tXfy/r2O727u7u9+d/fx7dtf++jdHR75jy9kxDtKc14dxfeaozFdP8GlOKu5EmrJFrxqY69paDmb+7e9oKgJvd7W/M+AMAZVW/lbWSFe51YubSB/EuEdlTevOreNZsETuKZCkryg2fQ0sypLCL1XqgjoBLXF1pVKsbDqDVhGFW/1pFshnK5C+LIO8ifh3VE686qbOdEsTOCppWoY0qTRc41x2OqlKFiTfTlx4KBppAWrSYNCcZAYjUltK3jxIni3aCF/CvS6o46uq5PybuwvUuUyhE1U9WiyqiQfbJwqQ81STGKXUWLDAWxMvkD2K9waUwu1bgX6ZT3kr6GJLwJ+R4FblwK3zUMArEMTFa9VEMzJSEn2O4g8g6/4aLAnMf7ou0K9g7OfplfXxGUr4I+opLf3//PmQe5+0N8G8x2VaV0q06o8fJDdMG49Zg1GXKDZSp8x+0DvZv4621L3kU5RR2GYbIyx5pJxhL7V5W+G8hTzH29/+PG3BH1HSVqXktQTdeoTzFxyzEwYnZLAUPO0iizSPWHTGYnERJBJVk4M3scXStnsZq098Zxbb2oL//fu3G/vfi/t/ic1J//wTt7+lnbZUYbWpQytppsa1RBmbLkPE0I+/dn10KgzFpOiNWU/WQeOWJJtgemtEhFz0BJ4Mwd0ToY+g/t3r2T0W53+1B3FbV2JW20ZS20WEsxyw2ynzWgS98KITaCrj+801UUZxIiRZqOpPXaP48ZJtyJJ8ay4XR8FvaiW8CTgO4rbuqojjxmMCmm2SEF9eAUthsO4OSP/RQRCNjpqkstgzz4X0LstJu90k4LoRm1dDXC4lJUuaNMLvdCOqrWuC8hjD2MmT4cNbNVn/GpLoaRhPMdWf8zj0AO+jZpGFGmg5qsGJ8wYeKuQHOPLvdAzs11OZiovnO1y0kA7it66Er1QLCSUjt6dvM7kd0hDC0WwcPSCzqzV+Ggy3jQ7oEVwb+pUa4OmPpZxKwOlVxro/r0+yOP9w4UGOvrxKwy0o2yuq/LyephMXuLwude1Y6rgXUrMAalKTsa0zDyzhwhIpXif+YgjgjT1vshbnYReO9vlaxOeMNsvs12uP665aLbLCdt5r5u9bGfPXnWkbNk8G5JiLT1zyMVTSdVvW/eQK7J38k3GhTFTbxnMmqOxN/jtQj1uZTs6y42Xe+tYoe6Z968LQd4IZz8jrG5tB/CGSioQwyjFRGKOktSUuPdEYUViFGNSmkdSwNbszeANxOEwf3MzSpUvMMLrp7ucBDzuCPhShhfEYKwJLI5n4SahlgDCsRFy74ZEBfRsCJlFGDnVOqepQYO8Vo15M8DLBYC/arzLSazTjliv+ocV4p6mkaowc1T0RjOl1kjdJyOPatwLYosjD/M6XHKmNnGa1ku5eBp2I6z//y5MjwF3hHx1pVqzaWpvz2biuRPlpFmGt5jxOUdkRvDul4FnsxBtuk3NuY9KmSlran0rSntFYfrTtb9hXfpLH36sLP2rZ317nflpR/Ovq9KhzqAxdePJ3ayOUX10VZyp1zKA++zm6ZLEakGm2EuFdIYRTVYC6WY77klV+nNH0K8sSl9UeFxn3CO/1ZWmzTuadpkLyJgs6ucIWSbMaUqHfdblsJ1sTlMbedeEbKEMqVUeqDkJKHQqzduMb6WFUrrEtGf96UEW+SY1zD597OOz1YTXHIrN+4ebxx/15ofbn/TudL3V+69W1iXHkTGUHY29zExAnikZwcveLWD6DGVz0UbRk9DgWaJfnU7Vtq2RwRZqMM0bO2Xiljhx2Spv/aKBPl8BfrGAOj3Q56QxeEdj8GogIMk0VcTaJpjOZVRiHqZlu7nMDk0Og5Zmbt59BxuCzsHkw7VUU9omCxdXXZDtH3vjQv2P9/eOzyIT8em9JdpHXjtBHP/w6cnff37yEvW6I+qrtt4Te5DBEpVk+K1f8mvqWHIJ5t6ajyXBHPNEb1Obhnm/Lq17rhS6+bytUD911eCzAb6knNd3DC40wgU3C54zCOyYL4DV4HGAJDgNZ286a8xd1Gt4lEdwwTpMSGEz1pEqe018TFETRW+53zSZvN3KIBdWva82yHGbXFPp/qzxrixyf9akO2YfANb93Y04VNtIOHw4oFZ2iyVIcbQyR2Bvpg29qWljle7dVrLWop1mssizlUnpxZ7tqxef1rU/swUvqGZ/1kY7JixgNd/cuw8nNabn7YQIAUBtDza/1VZSnYo1R2x6yBpFoB5Ncg/hWcX+adCtbHS6ov1ST3dRHfuzqO+YuoBl6iLDrCEYmGXOmalanK+FpM+oFUxLZa4orUTE0OfwcZrMMsmzRDAAt0J9NcXvE6jrnbFrLW+EHZMXsExe9ATDZ1WClplialNLZykGsTGvUDpQHuZ5es/VR9Ui8syJ/I7VRC/Z2wj0eBHovy78L2+96Jj52QW/Y+YAlpkDqQ28FZAt5IhzqFfmmaooYWQ/valQVLyZBzQLCx3DHKkZ320kwKobjWI27Ok67I8dBpx+6yUm2FHhw1Lh19hAnDJh8d4cwdvRcw9lhphKUOqljGpiLzPnEtW75mZTGxgz25+cNjJBumj575sxhR2lNiylNmfKgWo1eRGNw3q6xMSbCek+YEwT2Dr8QLPWMCRk8QlRpReJlScH3ai/p6F+xVDLoybZcrblK37A0RGXy+ddl2KDHYU+LIU+S88tQ9QOPqJv5OhZuGgejwvECo1ACE3QmPy3nRp0ejvSaAwAep2bBf0joy7PBP7XTrxcPf7awZfHf7srzbxjZgGWmQXkLhxaIGMWPAskBhzgm1vm5O5lt5hT6EayY+QWFE3VxjHj6M3cMWxk5hcl11bi6MPzKud0Wu1IyPu//j9+qInLVlEBAA=="""
rows=json.loads(gzip.decompress(base64.b64decode(payload)).decode())
TEMPLATE_REPLACEMENTS = {
    "dq4_omop_20260825_r5": RUN,
    "dq4_omop_20260825_r2": RUN,
    "2026-08-26T06:48:21.102Z": RUN_OPEN_TS,
    "cc682c9c-8795-4c48-adea-f988320f8d0d": SOURCE_UPDATE_ID,
    "f5c7c7ab-e37d-4a31-b9c2-b7631becb16a": SILVER_UPDATE_ID,
    "r5q9n3k6": SCRATCH_PREFIX,
}
def adapt(value):
    if isinstance(value, str):
        for old, new in TEMPLATE_REPLACEMENTS.items():
            value = value.replace(old, new)
        return value
    if isinstance(value, list):
        return [adapt(v) for v in value]
    if isinstance(value, dict):
        return {k: adapt(v) for k, v in value.items()}
    return value
rows = adapt(rows)
def q(v):
    return "'" + str(v).replace("'", "''") + "'"
def execute(seq,name,sql,missing_ok=False):
    sha=hashlib.sha256(sql.encode()).hexdigest()
    spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log
      (run_id,lane,stmt_file,seq,stmt_sha256,status,error_text,rows_affected,attempted_at,settled_at,created_by_session)
      VALUES ({q(RUN)},{q(LANE)},{q(name)},{seq},{q(sha)},'attempted',NULL,NULL,current_timestamp(),NULL,'DQ4')""")
    try:
        spark.sql(sql).collect()
        status="ok"; err="NULL"
    except Exception as e:
        msg=str(e)[:4000]
        if missing_ok and ("TABLE_OR_VIEW_NOT_FOUND" in msg or ("table or view" in msg.lower() and "cannot be found" in msg.lower())):
            status="skipped_missing_table"; err=q(msg)
        else:
            spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log
              (run_id,lane,stmt_file,seq,stmt_sha256,status,error_text,rows_affected,attempted_at,settled_at,created_by_session)
              VALUES ({q(RUN)},{q(LANE)},{q(name)},{seq},{q(sha)},'error',{q(msg)},NULL,current_timestamp(),current_timestamp(),'DQ4')""")
            raise
    spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log
      (run_id,lane,stmt_file,seq,stmt_sha256,status,error_text,rows_affected,attempted_at,settled_at,created_by_session)
      VALUES ({q(RUN)},{q(LANE)},{q(name)},{seq},{q(sha)},{q(status)},{err},NULL,current_timestamp(),current_timestamp(),'DQ4')""")
for row in rows:
    execute(row["seq"],row["path"],row["sql"],False)